In [44]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')


In [39]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')   

In [40]:
x_train = train_df.drop(columns=['Activity'])
y_train = train_df['Activity']
x_test = test_df.drop(columns=['Activity'])
y_test = test_df['Activity']

In [41]:
print('Calculating corealtion matrix')
corr_matrix =  x_train.corr().abs()

print('selecting upper triangle')
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))

print('removing highly corelated columns ')
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

final_x_train = x_train.drop(columns=to_drop)

final_x_test = x_test.drop(columns=to_drop)

Calculating corealtion matrix
selecting upper triangle
removing highly corelated columns 


In [42]:
scaler = StandardScaler()
x_train_scaled =  scaler.fit_transform(final_x_train)
x_test_scaled = scaler.fit_transform(final_x_test)
lb = LabelEncoder()
y_train_encoded  = lb.fit_transform(y_train)
y_test_encoded = lb.fit_transform(y_test)

In [43]:
np.save('X_train_scaled.npy', x_train_scaled)
np.save('X_test_scaled.npy', x_test_scaled)
np.save('y_train.npy', y_train_encoded)
np.save('y_test.npy', y_test_encoded)

In [45]:
models = {
    'RandomForestClassifier': RandomForestClassifier(),
    'ExtraTreesClassifier': ExtraTreesClassifier(), 
}
parameters = {
    'RandomForestClassifier': {
        
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'ExtraTreesClassifier': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }       
}

In [46]:
best_models = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    param_grid = parameters[model_name]
    random_search = RandomizedSearchCV(model, param_distributions=param_grid, n_iter=7, cv=5, n_jobs=-1, random_state=42, scoring='f1_macro')
    random_search.fit(x_train_scaled, y_train_encoded)
    best_models[model_name] = random_search.best_estimator_
    print(f"Best parameters for {model_name}: {random_search.best_params_}")

Training RandomForestClassifier...
Best parameters for RandomForestClassifier: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 10, 'bootstrap': False}
Training ExtraTreesClassifier...
Best parameters for ExtraTreesClassifier: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20, 'bootstrap': False}


In [47]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Testing data load ho raha hai...")
X_test = np.load('X_test_scaled.npy')
y_test = np.load('y_test.npy')



results = []

print("\n--- MODELS EVALUATION STARTED ---")
for name, model in best_models.items():
    print(f"\nEvaluating: {name}...")
    
    y_pred = model.predict(X_test)
    
    if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        y_pred = np.argmax(y_pred, axis=1)
    elif len(y_pred.shape) > 1:
        y_pred = y_pred.ravel()
        
    acc = accuracy_score(y_test, y_pred)
    f1_mac = f1_score(y_test, y_pred, average='macro')
    
    results.append({
        'Model Name': name,
        'Accuracy': round(acc * 100, 2),
        'F1-Score (Macro)': round(f1_mac * 100, 2)
    })
    
    print(f"--- {name} Classification Report ---")
    print(classification_report(y_test, y_pred))

print("\n" + "="*50)
print("FINAL PERFORMANCE COMPARISON TABLE")
print("="*50)

df_results = pd.DataFrame(results)
if not df_results.empty:
    df_results = df_results.sort_values(by='F1-Score (Macro)', ascending=False).reset_index(drop=True)
    print(df_results)
else:
    print("Koi model dictionary mein nahi mila. Pehle trained models ko add karein.")

Testing data load ho raha hai...

--- MODELS EVALUATION STARTED ---

Evaluating: RandomForestClassifier...
--- RandomForestClassifier Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       537
           1       0.86      0.94      0.90       491
           2       0.94      0.86      0.90       532
           3       0.96      0.99      0.98       496
           4       0.97      0.92      0.94       420
           5       0.93      0.94      0.94       471

    accuracy                           0.94      2947
   macro avg       0.94      0.94      0.94      2947
weighted avg       0.94      0.94      0.94      2947


Evaluating: ExtraTreesClassifier...
--- ExtraTreesClassifier Classification Report ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       537
           1       0.93      0.87      0.90       491
           2       0.89      0.94      0.92   

In [49]:
final_model = best_models['ExtraTreesClassifier']
import joblib
joblib.dump(final_model, 'final_model.pkl')

['final_model.pkl']